# Quy trình huấn luyện và tiếp tục checkpoint

Nguyễn Ngọc Hoàng Nam — B23DCCN585 | ASG 06

Các cell nhỏ được sắp theo thứ tự phụ thuộc. Huấn luyện đầy đủ mặc định tắt; số liệu chạy thử không phải kết quả test chính thức.

In [1]:
from pathlib import Path
import sys, json, os
ROOT=Path.cwd().resolve()
if ROOT.name=="notebooks": ROOT=ROOT.parent
assert (ROOT/"src").is_dir()
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.paths import DATA
from src.io import read_json
import numpy as np
import pandas as pd
from IPython.display import display, Image
print("Project:", ROOT)
print("Data:", DATA)


Project: E:\PTHTTM\ASG_06
Data: E:\PTHTTM\ASG_06_data


Bốn cấu hình được chạy lần lượt. PAUSE_TRAINING.bat yêu cầu dừng sau epoch hiện tại. Sau khi cửa sổ báo đã lưu và dừng, có thể Sleep. RUN_TRAINING.bat tiếp tục từ epoch gần nhất.

In [2]:
display(read_json(ROOT/'configs/main.json'))

{'seed': 42,
 'hidden_size': 32,
 'batch_size': 256,
 'max_epochs': 20,
 'patience': 4,
 'min_delta': 1e-05,
 'learning_rate': 0.001,
 'adam_epsilon': 1e-07,
 'clip_norm': 1,
 'sequence_lengths': {'retailrocket': 20, 'sp500': 30},
 'frameworks': ['pytorch', 'keras'],
 'datasets': ['retailrocket', 'sp500']}

## Trạng thái thực nghiệm

In [3]:
rows=[]
for name in ['retailrocket','sp500']:
    for framework in ['pytorch','keras']:
        folder=ROOT/'results/runs'/f'{name}_{framework}_seed42'
        state=read_json(folder/'state.json') if (folder/'state.json').exists() else {}
        rows.append([name,framework,state.get('epoch',0),(folder/'metrics.json').exists()])
display(pd.DataFrame(rows,columns=['dataset','framework','saved epoch','completed']))

,dataset,framework,saved epoch,completed
0,retailrocket,pytorch,0,False
1,retailrocket,keras,0,False
2,sp500,pytorch,0,False
3,sp500,keras,0,False


## Bằng chứng chạy tiếp
So sánh chạy liền hai epoch với dừng sau epoch một rồi khởi động lại; dữ liệu chỉ là smoke subset.

In [4]:
path=ROOT/'results/resume_verification.json'
if path.exists(): display(read_json(path))

[{'framework': 'pytorch',
  'process_restart': True,
  'max_output_error': 0.0,
  'passed': True},
 {'framework': 'keras',
  'process_restart': True,
  'max_output_error': 0.0,
  'passed': True}]